# Chapter 10 — Attention: Which Position Is Comparing With Which?

**Book alignment:** PyTorch From First Principles, Chapter 10

**Question this notebook isolates:** Does each query row form a valid distribution over the intended keys (rows sum to 1, forbidden weights 0, causal prefix invariant) with head rearrangement that round-trips exactly?

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(1)
print('torch', torch.__version__)

## 1 — Head split must partition features, and round-trip exactly

`reshape(B, Nh, T, Dh)` has the right shape but splits the sequence; the correct split is `reshape(B, T, Nh, Dh).transpose(1, 2)`. Only the correct one round-trips.

In [ ]:
def split_heads(x, nh):
    B, T, E = x.shape
    assert E % nh == 0
    return x.reshape(B, T, nh, E // nh).transpose(1, 2)

def merge_heads(x):
    B, Nh, T, Dh = x.shape
    return x.transpose(1, 2).reshape(B, T, Nh * Dh)

B, T, E, Nh = 1, 4, 8, 2
x = torch.arange(B * T * E).reshape(B, T, E)
bad = x.reshape(B, Nh, T, E // Nh)
good = split_heads(x, Nh)
print('shape equal:', tuple(bad.shape) == tuple(good.shape))
print('values equal:', bool(torch.equal(bad, good)))
rt_good = merge_heads(good)
rt_bad = bad.transpose(1, 2).reshape(B, T, E)
print('good round trip:', bool(torch.equal(rt_good, x)))
print('bad round trip :', bool(torch.equal(rt_bad, x)))

In [ ]:
assert tuple(bad.shape) == tuple(good.shape) == (1, 2, 4, 4)
assert not torch.equal(bad, good)
assert torch.equal(rt_good, x)
assert not torch.equal(rt_bad, x)
print('head split semantics verified')

## 2 — Softmax over keys gives rows that sum to 1; manual attention matches SDPA

Softmax over the key axis (`dim=-1`) normalizes each query row; over queries (`dim=-2`) it does not. A hand-written causal attention must agree with `F.scaled_dot_product_attention`.

In [ ]:
def manual_attention(q, k, v, allow_mask=None):
    Dh = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(Dh)
    masked = scores if allow_mask is None else scores.masked_fill(~allow_mask, float('-inf'))
    w = torch.softmax(masked, dim=-1)
    return w @ v, {'scores': scores, 'weights': w}

torch.manual_seed(2)
q = torch.randn(2, 2, 4, 4)
k = torch.randn(2, 2, 4, 4)
v = torch.randn(2, 2, 4, 4)
Tt = 4
allowed = torch.ones(Tt, Tt, dtype=torch.bool).tril()

mine, st = manual_attention(q, k, v, allowed)
theirs = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=0.0)
diff = float((mine - theirs).abs().max())
print('manual vs SDPA max diff:', diff)
print('row-sum error:', float((st['weights'].sum(-1) - 1).abs().max()))
print('forbidden max:', float(st['weights'][..., ~allowed].abs().max()))

raw = torch.randn(1, 2, 4, 4)
good_w = torch.softmax(raw, dim=-1)
bad_w = torch.softmax(raw, dim=-2)
good_err = float((good_w.sum(-1) - 1).abs().max())
bad_vals = (bad_w.sum(-1) - 1).abs()
print('good axis err:', good_err, 'bad axis min dev:', float(bad_vals.max()))

In [ ]:
assert diff < 1e-5
assert float((st['weights'].sum(-1) - 1).abs().max()) < 1e-5
assert float(st['weights'][..., ~allowed].abs().max()) < 1e-6
assert good_err < 1e-5
assert float(bad_vals.max()) > 1e-3
print('softmax axis + SDPA agreement verified')

## 3 — Causality is behavioral: future edits must not move the past

A triangular mask is a claim. The evidence: change tokens after position `i` and require prefix outputs unchanged while suffix outputs move.

In [ ]:
class TinyAttn(nn.Module):
    def __init__(self, e=8, nh=2):
        super().__init__()
        self.nh = nh
        self.qkv = nn.Linear(e, 3 * e, bias=False)
        self.proj = nn.Linear(e, e, bias=False)
    def forward(self, x, causal=True):
        B, T, E = x.shape
        Dh = E // self.nh
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.reshape(B, T, self.nh, Dh).transpose(1, 2)
        k = k.reshape(B, T, self.nh, Dh).transpose(1, 2)
        v = v.reshape(B, T, self.nh, Dh).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        return self.proj(y.transpose(1, 2).reshape(B, T, E))

torch.manual_seed(3)
attn = TinyAttn().eval()
B, T, E, i = 2, 6, 8, 2
x = torch.randn(B, T, E)
x2 = x.clone()
x2[:, i + 1:] = torch.randn(B, T - i - 1, E)
with torch.no_grad():
    y1 = attn(x, causal=True)
    y2 = attn(x2, causal=True)
    y3 = attn(x, causal=False)
    y4 = attn(x2, causal=False)
pre = float((y1[:, :i + 1] - y2[:, :i + 1]).abs().max())
suf = float((y1[:, i + 1:] - y2[:, i + 1:]).abs().max())
pre_nc = float((y3[:, :i + 1] - y4[:, :i + 1]).abs().max())
print(f'causal prefix delta={pre:.3e} suffix delta={suf:.3e}')
print(f'non-causal prefix delta={pre_nc:.3e}')

In [ ]:
assert pre < 1e-6
assert suf > 1e-4
assert pre_nc > 1e-4
print('causal intervention verified')

## What we earned

Attention can hold the right shape while comparing the wrong objects. The checks that see it are value-level: head round-trip, key-axis row sums, zero forbidden weights, reference agreement, and a future-token intervention.

Chapter 11 asks the next question: when every one of those passes yet the model does not learn, which link in the learning chain is missing?